# Preprocesamiento de FracAtlas
Este notebook aplica el pipeline de mejora de imagen (CLAHE + Mediana + Unsharp Masking) a todas las imagenes del dataset FracAtlas.

In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil

In [2]:
def get_safe_path(path):
    """Convierte rutas largas a formato seguro para Windows si es necesario."""
    abs_path = os.path.abspath(path)
    if os.name == 'nt' and not abs_path.startswith('\\\\?\\'):
        return '\\\\?\\' + abs_path
    return abs_path

In [3]:
def aplicar_pipeline_balanceado(img_gray):
    """Pipeline extraido del EDA para mejorar el contraste y reducir ruido."""
    # 1. CLAHE (clipLimit=3.0)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)
    
    # 2. Filtro Mediana (3x3)
    img_median = cv2.medianBlur(img_clahe, 3)
    
    # 3. Unsharp Masking Moderado (k=1.2)
    gaussian_blur = cv2.GaussianBlur(img_median, (0,0), 1.0)
    img_enhanced = cv2.addWeighted(img_median, 2.2, gaussian_blur, -1.2, 0)
    
    return img_enhanced

In [4]:
input_dir = r'..\data\FracAtlas\images'
output_dir = r'..\data\FracAtlas_procesado\images'

# Crear directorios de salida
for class_name in ['Fractured', 'Non_fractured']:
    os.makedirs(get_safe_path(os.path.join(output_dir, class_name)), exist_ok=True)

In [5]:
# Procesar todas las imagenes
total_images = 0
processed_images = 0
errors = 0

for class_name in ['Fractured', 'Non_fractured']:
    class_input_dir = get_safe_path(os.path.join(input_dir, class_name))
    class_output_dir = get_safe_path(os.path.join(output_dir, class_name))
    
    if not os.path.exists(class_input_dir):
        print(f"Advertencia: No se encontro la carpeta {class_input_dir}")
        continue
        
    image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    total_images += len(image_files)
    
    print(f"Procesando carpeta {class_name} ({len(image_files)} imagenes)...")
    
    for img_name in tqdm(image_files):
        in_path = os.path.join(class_input_dir, img_name)
        out_path = os.path.join(class_output_dir, img_name)
        
        try:
            # Leer imagen correctamente en Windows
            with open(in_path, 'rb') as f:
                img_array = np.frombuffer(f.read(), np.uint8)
            img_orig = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
            
            if img_orig is not None:
                # Aplicar pipeline
                img_proc = aplicar_pipeline_balanceado(img_orig)
                
                # Guardar imagen procesada
                success, encoded_img = cv2.imencode('.jpg', img_proc)
                if success:
                    with open(out_path, 'wb') as f:
                        encoded_img.tofile(f)
                    processed_images += 1
                else:
                    print(f"Error al codificar imagen: {img_name}")
                    errors += 1
            else:
                print(f"Error al leer imagen: {img_name}")
                errors += 1
        except Exception as e:
            print(f"Excepcion procesando {img_name}: {e}")
            errors += 1
            
print(f"\nResumen:")
print(f"Total imagenes encontradas: {total_images}")
print(f"Imagenes procesadas exitosamente: {processed_images}")
print(f"Errores: {errors}")

Procesando carpeta Fractured (717 imagenes)...


100%|██████████| 717/717 [00:46<00:00, 15.27it/s] 


Procesando carpeta Non_fractured (3366 imagenes)...


100%|██████████| 3366/3366 [02:35<00:00, 21.71it/s]


Resumen:
Total imagenes encontradas: 4083
Imagenes procesadas exitosamente: 4083
Errores: 0
